### Parameters

In [ ]:
import re
from datetime import UTC, datetime

dbutils.widgets.text(
    "landing_path",
    "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/population",
    "Population landing folder",
)
dbutils.widgets.dropdown("reload_all", "false", ["false", "true"], "Reload loaded batches")

CATALOG = "public_health"
LANDING_PATH = dbutils.widgets.get("landing_path")
RELOAD_ALL = dbutils.widgets.get("reload_all") == "true"

TABLE = f"{CATALOG}.bronze.population_bronze"
N_COLS = 14  # c0..c13
CELL_COLS = [f"c{i}" for i in range(N_COLS)]
CELLS_SQL = ", ".join(CELL_COLS)  # c0, c1, ..., c13
RELOAD_SQL = "TRUE" if RELOAD_ALL else "FALSE"
# No backslashes on purpose: Spark SQL string literals would eat them ("\d" becomes "d")
FILE_REGEX = "^population_([0-9]{4})_asof_([0-9]{8})[.]xlsx$"


### Create Table

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS public_health.bronze.population_bronze (
    c0 STRING, c1 STRING, c2 STRING, c3 STRING, c4 STRING, c5 STRING, c6 STRING,
    c7 STRING, c8 STRING, c9 STRING, c10 STRING, c11 STRING, c12 STRING, c13 STRING,
    _rescued_data STRING,
    _source_file STRING,
    _source_sheet STRING,
    _source_row_num INT,
    _as_of_date DATE,
    _ingested_at TIMESTAMP
)
COMMENT 'Raw PSA population files, 1 batch per file, positional columns, no header parsing'


### File Loading

In [ ]:
landing = spark.createDataFrame(
    [(f.name,) for f in dbutils.fs.ls(LANDING_PATH)], "file_name STRING"
)
landing.createOrReplaceTempView("population_landing_files")

BATCHES_SQL = f"""
SELECT
  file_name,
  CAST(try_to_timestamp(regexp_extract(file_name, '{FILE_REGEX}', 2), 'yyyyMMdd') AS DATE)
    AS as_of_date
FROM population_landing_files
WHERE file_name RLIKE '{FILE_REGEX}'
  AND try_to_timestamp(regexp_extract(file_name, '{FILE_REGEX}', 2), 'yyyyMMdd') IS NOT NULL
  AND ({RELOAD_SQL}
       OR NOT EXISTS (
         SELECT 1 FROM {TABLE} AS b WHERE b._source_file = population_landing_files.file_name
       ))
ORDER BY as_of_date, file_name
"""
batches = spark.sql(BATCHES_SQL).collect()

ignored = [
    r.file_name
    for r in spark.sql(
        f"SELECT file_name FROM population_landing_files WHERE NOT file_name RLIKE '{FILE_REGEX}'"
    ).collect()
]
if ignored:
    print(f"Ignored (name does not follow the convention): {ignored}")
print(f"To load: {[(b.file_name, str(b.as_of_date)) for b in batches]}")

### Loading Each Batch

In [ ]:
SCHEMA_DDL = ", ".join(f"{c} STRING" for c in CELL_COLS)
SAFE_NAME = re.compile(r"^[A-Za-z0-9 _.\-]+$")  # sheet names are pasted here


def list_sheets(path: str) -> list[str]:
    rows = spark.sql(
        f"""SELECT sheetName FROM read_files('{path}', format => 'excel',
            schemaEvolutionMode => 'none', operation => 'listSheets') ORDER BY sheetIndex"""
    ).collect()
    return [r.sheetName for r in rows]


def check_width(path: str, sheet: str) -> None:
    """Fail loudly if the sheet is wider than N_COLS (nothing would rescue the extra cells)."""
    cols = spark.sql(
        f"""SELECT * FROM read_files('{path}', format => 'excel', headerRows => 0,
            dataAddress => "{sheet}", schemaEvolutionMode => 'none') LIMIT 0"""
    ).columns
    if len(cols) > N_COLS:
        raise ValueError(f"{path} [{sheet}] has {len(cols)} columns; raise N_COLS ({N_COLS})")


def load_sheet(file_name: str, as_of_date, sheet: str, ingested_at: str) -> None:
    path = f"{LANDING_PATH}/{file_name}"
    spark.sql(f"""
        INSERT INTO {TABLE}
        REPLACE WHERE _source_file = '{file_name}' AND _source_sheet = '{sheet}'
        SELECT
          {CELLS_SQL},
          CAST(NULL AS STRING) AS _rescued_data,
          '{file_name}' AS _source_file,
          '{sheet}' AS _source_sheet,
          _row_num AS _source_row_num,
          DATE'{as_of_date}' AS _as_of_date,
          TIMESTAMP'{ingested_at}' AS _ingested_at
        FROM (
          SELECT
            *,
            CAST(row_number() OVER (ORDER BY monotonically_increasing_id()) AS INT) AS _row_num
          FROM read_files(
            '{path}',
            format => 'excel',
            headerRows => 0,
            dataAddress => "{sheet}",
            schema => '{SCHEMA_DDL}',
            schemaEvolutionMode => 'none'
          )
        )
        WHERE coalesce({CELLS_SQL}) IS NOT NULL
    """)

In [ ]:
for b in batches:
    ingested_at = datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S+00:00")  # 1 timestamp per batch
    path = f"{LANDING_PATH}/{b.file_name}"
    for sheet in list_sheets(path):
        if not SAFE_NAME.match(sheet):
            raise ValueError(f"Unsupported characters in sheet name: {sheet!r}")
        check_width(path, sheet)
        load_sheet(b.file_name, b.as_of_date, sheet, ingested_at)
        print(f"Loaded {b.file_name} [{sheet}]")

if not batches:
    print("Nothing to load.")

### Checks

In [ ]:
%sql
SELECT 
    _source_file, 
    _source_sheet, 
    _as_of_date, 
    count(*) AS row_count,
    max(_ingested_at) AS last_ingested_at
FROM public_health.bronze.population_bronze
GROUP BY _source_file, _source_sheet, _as_of_date
ORDER BY _as_of_date, _source_file, _source_sheet

### Identifying the Header Row/s

In [ ]:
%sql
SELECT *
FROM public_health.bronze.population_bronze
ORDER BY _source_file, _source_sheet, _source_row_num
LIMIT 15

### Idempotency Check

In [ ]:
%sql
SELECT 
    _source_file, 
    _source_sheet, 
    _source_row_num,
    count(*) AS  copies
FROM public_health.bronze.population_bronze
GROUP BY _source_file, _source_sheet, _source_row_num
HAVING count(*) > 1 